# Fase 4
Para poder avanzar a la fase de Entrenamiento del Modelo, necesito que me confirmes cómo te fue con la transformación de los datos:

1. Eliminación de Columnas: ¿Qué columnas eliminaste? (Recuerda que name y rank no ayudan a predecir ventas numéricas, solo ensucian el modelo).

2. Variable de Antigüedad: ¿Cómo calculaste la columna years_since_launch? (¿Usaste el año máximo del dataset o el año actual 2026?).

3. La Explosión de Columnas: Tras aplicar pd.get_dummies() a genre y platform, ¿cuántas columnas tiene ahora tu DataFrame?

Empezamos importando dependecias y convirtiendo nuestro dataset en un dataframe.

In [1]:
import pandas as pd
from pathlib import Path

path = Path(r"C:\Users\antim\OneDrive\Escritorio\poryectos\proyecto_videojuegos\data\dataset_limpio.parquet")
path

WindowsPath('C:/Users/antim/OneDrive/Escritorio/poryectos/proyecto_videojuegos/data/dataset_limpio.parquet')

In [2]:
data = pd.read_parquet(path)

data.head()

,rank,name,platform,year,genre,publisher,na_sales,eu_sales,jp_sales,other_sales,global_sales
0,1,Wii Sports,Wii,2006,Sports,Nintendo,41.49,29.02,3.77,8.46,82.74
1,2,Super Mario Bros.,NES,1985,Platform,Nintendo,29.08,3.58,6.81,0.77,40.24
2,3,Mario Kart Wii,Wii,2008,Racing,Nintendo,15.85,12.88,3.79,3.31,35.82
3,4,Wii Sports Resort,Wii,2009,Sports,Nintendo,15.75,11.01,3.28,2.96,33.00
4,5,Pokemon Red/Pokemon Blue,GB,1996,Role-Playing,Nintendo,11.27,8.89,10.22,1.00,31.37


## 1 Eliminacion de columnas
Eleiminaremos las columnas que no sirvan para predecciones del modelo, es decir aquellas que no aporten datos caracteristicos relevantes.
En este eliminarmeos las columnas de 'name' y 'rank'.
Ademas de eliminar la columna de sales globales, ya que sino tendremos data leakage, debido a que nos interesa inferir venas a nivel global en base a genero plataforma y publisher.

In [3]:
df_modelo = data[['platform', 'year', 'genre', 'publisher', 'global_sales']]

df_modelo.head()

,platform,year,genre,publisher,global_sales
0,Wii,2006,Sports,Nintendo,82.74
1,NES,1985,Platform,Nintendo,40.24
2,Wii,2008,Racing,Nintendo,35.82
3,Wii,2009,Sports,Nintendo,33.00
4,GB,1996,Role-Playing,Nintendo,31.37


## 2 Elimiacion de registros
Luego viendo los datos faltantes apartir de notebooks anteriores luego del 2015 lo que haremos sera eliminar aquellas filas cuyos year sean mayores a 2015.

In [4]:
print(f"Cantidad de registros posteriores a 2015: {df_modelo[df_modelo['year'] > 2015].value_counts().sum()}")

# eliminamos los registros posterires a 2015
# df_modelo = df_modelo[df_modelo['year'] <= 2015] -> codigo en caso de querer eliminar registros posteriores al 2015

Cantidad de registros posteriores a 2015: 348


## 3 Creamos la fila antiguedad
Crearemos la columna 'years_on_market' que nos premitira saber hace cuanto esta un producot en el mercado, y luego eliminaremos la columna 'year'

In [5]:
#creamos la columna calculadnola en base a 2016
df_modelo['years_on_market'] = 2016 - df_modelo['year']

# eliminamos la columna year
df_modelo = df_modelo.drop(columns='year')

df_modelo.head()

,platform,genre,publisher,global_sales,years_on_market
0,Wii,Sports,Nintendo,82.74,10
1,NES,Platform,Nintendo,40.24,31
2,Wii,Racing,Nintendo,35.82,8
3,Wii,Sports,Nintendo,33.00,7
4,GB,Role-Playing,Nintendo,31.37,20


## 5 Eliminamos la columna publisher
En este primer modelo eliminaremos la columna publisher ya que tenemops mas de 500 registros diferentes, por lo que hacer un one-hot encoding crearia un problema de the curse of dimentonality.

In [6]:
# Primero veamos la cantidad de publisher que hay
print(f"Cantidad de 'publisher': {len(df_modelo['publisher'].value_counts())}")

# eliminamos la columna del dataframe
df_modelo = df_modelo.drop(columns='publisher')

df_modelo.head()

Cantidad de 'publisher': 576


,platform,genre,global_sales,years_on_market
0,Wii,Sports,82.74,10
1,NES,Platform,40.24,31
2,Wii,Racing,35.82,8
3,Wii,Sports,33.00,7
4,GB,Role-Playing,31.37,20


## 5: One-Hot Encoding de Categorías
Transformar el texto en binario (0 y 1).

Acción: Aplica pd.get_dummies() a las columnas platform y genre.

Parámetro Técnico: Usa drop_first=True dentro de la función. Esto evita la "trampa de la variable ficticia" (multicolinealidad), algo que los estadísticos de la mesa directiva te agradecerán.

En este caso utiliazremos el `OneHotEncoder`, el cual es la forma mas rigurosa de hacerlo

In [7]:
!pip install scikit-learn


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


# 1. Instanciamos el OneHotEncoder
encoder = OneHotEncoder(drop='first', sparse_output=False) # importante el spaarse en false

# 2. Configuramos el ColumnTransformer
preprocesador = ColumnTransformer(
    transformers=[
        # ('nombre_del_paso', transformador_a_usar, ['columnas_a_transformar'])
        ('codificacion_categorica', encoder, ['platform', 'genre'])
    ],
    remainder='passthrough' # Que hacer con el resto de columnas
)

# 3. Ajustamos y transformamos los datos
datos_transformados = preprocesador.fit_transform(df_modelo)

### 4. (Opcional pero recomendado) Reconstruimos el DataFrame para visualizarlo
# Obtenemos los nombres generados para las nuevas columnas categóricas       # este metodo nos devuele el nombre de las columnas nuevas
nombres_nuevos = preprocesador.named_transformers_['codificacion_categorica'].get_feature_names_out(['platform', 'genre'])

# Las columnas 'passthrough' siempre se colocan al final por defecto
nombres_todas_las_columnas = list(nombres_nuevos) + ['global_sales', 'years_on_market'] 

# Creamos el DataFrame final
df_modelo_onehotencoding = pd.DataFrame(datos_transformados, columns=nombres_todas_las_columnas)

In [9]:
df_modelo_onehotencoding.columns

Index(['platform_3DO', 'platform_3DS', 'platform_DC', 'platform_DS',
       'platform_GB', 'platform_GBA', 'platform_GC', 'platform_GEN',
       'platform_GG', 'platform_N64', 'platform_NES', 'platform_NG',
       'platform_PC', 'platform_PCFX', 'platform_PS', 'platform_PS2',
       'platform_PS3', 'platform_PS4', 'platform_PSP', 'platform_PSV',
       'platform_SAT', 'platform_SCD', 'platform_SNES', 'platform_TG16',
       'platform_WS', 'platform_Wii', 'platform_WiiU', 'platform_X360',
       'platform_XB', 'platform_XOne', 'genre_Adventure', 'genre_Fighting',
       'genre_Misc', 'genre_Platform', 'genre_Puzzle', 'genre_Racing',
       'genre_Role-Playing', 'genre_Shooter', 'genre_Simulation',
       'genre_Sports', 'genre_Strategy', 'global_sales', 'years_on_market'],
      dtype='str')

In [10]:
df_modelo_onehotencoding.head()

,platform_3DO,platform_3DS,platform_DC,platform_DS,platform_GB,platform_GBA,platform_GC,platform_GEN,platform_GG,platform_N64,...,genre_Platform,genre_Puzzle,genre_Racing,genre_Role-Playing,genre_Shooter,genre_Simulation,genre_Sports,genre_Strategy,global_sales,years_on_market
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,82.74,10.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.24,31.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,35.82,8.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,33.00,7.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,31.37,20.0


## 6 Definimos variables independientes y variabel dependiente.

definiremos una matriz con las variabels independientes que en este caso seran todas menos 'gloabal_sales', y luuego nuestra variable dependiente que sera 'gloabal_sales'

In [11]:
# Defimos nuestra matriz X
X = df_modelo_onehotencoding.drop(columns='global_sales')
# Defininmos nuestra y
y = df_modelo_onehotencoding['global_sales']


X.shape, y.shape

((16324, 42), (16324,))

Ya habiendo definido nuestra matriz y nuestra variable dependiente. Podemos guardar el daataFrame, dentro del directorio de datasetes, de esta forma lo tenderemso listo para la proxima vez que lo queramos usar.

In [14]:
import sys
import os

sys.path.append(os.path.abspath("..")) # esto sirve para que python tenga disponible nuestra carpeta src

In [15]:
# utilizaremos uns funcion de la carpeta src 
from src.funciones import guardar_parquet

guardar_parquet(
  r"C:\Users\antim\OneDrive\Escritorio\poryectos\proyecto_videojuegos\data\data_set_ml_01.parquet",
  df_modelo_onehotencoding
)